In [1]:
import json
import time
import requests
import pandas as pd

with open("retrieval_config.json", "r") as f:
    config = json.load(f)

API_KEY = config["API_KEY"]
URL_OPERATIONAL = "https://api.eia.gov/v2/electricity/electric-power-operational-data/data/"

def fetch_all_eia_data(api_key, page_size=5000, sleep_seconds=0.2):
    all_rows = []
    offset = 0
    total_available = None
    session = requests.Session()

    while True:
        params = [
            ("api_key", api_key),
            ("frequency", "monthly"),
            ("data[0]", "generation"),

            # Keep your existing sector filters
            ("facets[sectorid][]", "8"),
            ("facets[sectorid][]", "96"),
            ("facets[sectorid][]", "97"),

            # Add your requested filters
            ("facets[location][]", "NC"),
            ("facets[fueltypeid][]", "ALL"),

            # Keep your sort order
            ("sort[0][column]", "fueltypeid"),
            ("sort[0][direction]", "asc"),
            ("sort[1][column]", "period"),
            ("sort[1][direction]", "asc"),

            # Pagination
            ("offset", offset),
            ("length", page_size),
        ]
        try:
            response = session.get(URL_OPERATIONAL, params=params, timeout=60)

            if response.status_code == 400:
                print("400 error from API:")
                print(response.text)
                return pd.DataFrame()

            response.raise_for_status()
            payload = response.json()

            if "response" not in payload or "data" not in payload["response"]:
                print("Unexpected response format:")
                print(payload)
                return pd.DataFrame()

            batch = payload["response"]["data"]

            if total_available is None:
                total_available = int(payload["response"]["total"])
                print(f"Total matching rows: {total_available:,}")

            if not batch:
                break

            all_rows.extend(batch)
            print(f"Fetched {len(all_rows):,} / {total_available:,} rows")

            if len(all_rows) >= total_available:
                break

            offset += page_size
            time.sleep(sleep_seconds)

        except requests.exceptions.RequestException as e:
            print(f"Request error: {e}")
            break
        except ValueError as e:
            print(f"JSON parse error: {e}")
            break
        except Exception as e:
            print(f"Unexpected error: {e}")
            break

    return pd.DataFrame(all_rows)

df_nc_generation = fetch_all_eia_data(API_KEY, page_size=5000)

if not df_nc_generation.empty:
    if "generation" in df_nc_generation.columns:
        df_nc_generation["generation"] = pd.to_numeric(
            df_nc_generation["generation"], errors="coerce"
        )

    desired_cols = [
        "chart",
        "period",
        "location",
        "stateDescription",
        "sectorid",
        "sectorDescription",
        "fueltypeid",
        "fuelTypeDescription",
        "generation",
        "generation-units",
    ]

    existing_cols = [c for c in desired_cols if c in df_nc_generation.columns]
    df_nc_generation = df_nc_generation[existing_cols]

    print(f"\nSuccess! Retrieved {len(df_nc_generation):,} rows.")
    print(df_nc_generation.head())

    df_nc_generation.to_csv("nc_generation_all.csv", index=False)
    print("Saved to nc_generation_all.csv")

else:
    print("\nNo data returned. Check the API key and query parameters.")

Total matching rows: 602
Fetched 602 / 602 rows

Success! Retrieved 602 rows.
    period location stateDescription sectorid sectorDescription fueltypeid  \
0  2001-01       NC   North Carolina       96    All Commercial        ALL   
1  2001-01       NC   North Carolina       97    All Industrial        ALL   
2  2001-02       NC   North Carolina       96    All Commercial        ALL   
3  2001-02       NC   North Carolina       97    All Industrial        ALL   
4  2001-03       NC   North Carolina       96    All Commercial        ALL   

  fuelTypeDescription  generation        generation-units  
0           all fuels      13.646  thousand megawatthours  
1           all fuels     303.266  thousand megawatthours  
2           all fuels      11.036  thousand megawatthours  
3           all fuels     269.517  thousand megawatthours  
4           all fuels       7.545  thousand megawatthours  
Saved to nc_generation_all.csv
